# Resultados por Dataset: RPD e Desvio Padrão

Uma tabela por dataset (A, B, X). Para cada instância e algoritmo, usam-se as **30 seeds**.

- **RPD (%)** = `(best − média_obj) / best · 100` (maximização), com `best` de `best_solutions/best_objectives.csv`.
- **Desvio padrão** = desvio padrão amostral (`ddof=1`) do objetivo entre as seeds.

In [13]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, HTML

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks" / "utils" / "__init__.py").exists())
RESULTS = ROOT / "project" / "results"
BEST_CSV = ROOT / "best_solutions" / "best_objectives.csv"

DATASETS = ["a", "b", "x"]
ALGOS = {"order_based": "OrderBased", "aisle_based": "AisleBased",
         "ils_vnd_all": "ILS-VND", "andre_feijo_all": "Santos e Baldotto (2025)"}

import matplotlib
matplotlib.use("Agg") if False else None
import matplotlib.pyplot as plt

IMG = ROOT / "notebooks" / "resultados" / "img"
IMG.mkdir(parents=True, exist_ok=True)

def salvar_tabela(txt, titulo, nome, cores=None, dpi=200):
    """Salva um DataFrame de strings como imagem PNG em IMG/nome.png."""
    nl, nc = txt.shape
    fig, ax = plt.subplots(figsize=(1.6 * nc + 1.2, 0.32 * (nl + 2) + 0.6))
    ax.axis("off")
    ax.set_title(titulo, fontsize=10, pad=8)
    tb = ax.table(cellText=txt.values, rowLabels=[str(i) for i in txt.index],
                  colLabels=list(txt.columns), cellLoc="center", loc="center")
    tb.auto_set_font_size(False); tb.set_fontsize(8); tb.scale(1, 1.2)
    tb.auto_set_column_width(list(range(nc)))
    for (r, c), cell in tb.get_celld().items():
        cell.set_edgecolor("#999")
        if r == 0 or c == -1:
            cell.set_facecolor("#eeeeee"); cell.set_text_props(weight="bold")
        elif cores is not None and cores.iloc[r - 1, c]:
            cell.set_text_props(color=cores.iloc[r - 1, c])
    fig.savefig(IMG / f"{nome}.png", dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)

def salvar_diff(d, titulo, nome):
    cores = d.map(lambda v: "#1a9c00" if v > 1e-9 else ("#d00000" if v < -1e-9 else ""))
    salvar_tabela(d.map("{:+.2f}".format), titulo, nome, cores)


In [14]:
frames = []
for folder, label in ALGOS.items():
    for ds in DATASETS:
        for f in sorted((RESULTS / folder / ds).glob("instance_*.csv")):
            d = pd.read_csv(f, usecols=["run_id", "seed", "status", "objective"])
            d["algo"], d["dataset"], d["instance"] = label, ds, f.stem + ".txt"
            frames.append(d)
raw = pd.concat(frames, ignore_index=True)

best = pd.read_csv(BEST_CSV).rename(columns={"best_objective": "best"})
raw = raw.merge(best, on=["dataset", "instance"], how="left")

# Sanidade
print("Status:", raw.status.value_counts().to_dict())
print("Objetivos NaN:", int(raw.objective.isna().sum()), "| sem best:", int(raw.best.isna().sum()))
print(raw.groupby(["dataset", "algo"]).size().unstack().assign())
seeds = raw.groupby(["dataset", "instance", "algo"]).size()
print("Instância/algoritmo com != 30 seeds:", int((seeds != 30).sum()))

Status: {'feasible': 6000}
Objetivos NaN: 0 | sem best: 0
algo     AisleBased  Santos e Baldotto (2025)  GVNS  OrderBased
dataset                                           
a               600          600   600         600
b               450          450   450         450
x               450          450   450         450
Instância/algoritmo com != 30 seeds: 0


In [15]:
agg = (raw.groupby(["dataset", "instance", "algo"])
          .agg(media=("objective", "mean"), dp=("objective", lambda s: s.std(ddof=1)), best=("best", "first"))
          .reset_index())
agg["rpd"] = (agg.best - agg.media) / agg.best * 100

ORDEM = list(ALGOS.values())

def tabela(ds):
    d = agg[agg.dataset == ds]
    out = {}
    for metrica, col in [("RPD (%)", "rpd"), ("Desvio Padrão", "dp")]:
        p = d.pivot(index="instance", columns="algo", values=col)[ORDEM]
        p.index = [int(i.split("_")[1].split(".")[0]) for i in p.index]
        out[metrica] = p
    t = pd.concat(out, names=["Métricas ↓", "Instância"])
    t.columns.name = "Algoritmos →"
    return t

def mostrar(t, titulo):
    rpd, dp = t.loc["RPD (%)"], t.loc["Desvio Padrão"]
    s = rpd.map("{:.2f}".format) + " ± " + dp.map("{:.2f}".format)
    s = s.mask(dp.abs() < 1e-9, rpd.map("{:.2f}".format))  # DP zero: mostra so o RPD
    s.index.name = "Instância"
    s.columns.name = "Algoritmos → (RPD % ± DP)"
    display(s.style.set_table_styles([
                {"selector": "th", "props": [("text-align", "center"), ("border", "1px solid #999")]},
                {"selector": "td", "props": [("text-align", "center")]}])
             .set_caption(titulo))
    return s

for ds in DATASETS:
    t = tabela(ds)
    titulo = f"Dataset {ds.upper()}: RPD (%) ± desvio padrão do objetivo (30 seeds)"
    s = mostrar(t, titulo)
    salvar_tabela(s, titulo, f"tabela_rpd_dp_{ds}")


Algoritmos → (RPD % ± DP),OrderBased,AisleBased,GVNS,Santos e Baldotto (2025)
Instância,,,,
1,65.13,0.00,0.00,0.00
2,0.00,0.00,0.00,0.00
3,61.59,29.17,0.00,0.00
4,39.85,0.00,0.00,0.00
5,80.23,3.09,0.00,0.00
6,95.38,0.00,0.00,0.00
7,92.68,0.00,0.00,0.00
8,80.20,10.01,0.65 ± 0.38,0.18
9,33.38,31.17,0.26 ± 0.06,0.00


Algoritmos → (RPD % ± DP),OrderBased,AisleBased,GVNS,Santos e Baldotto (2025)
Instância,,,,
1,95.20,0.00,0.00,0.00
2,80.77,3.90,0.05 ± 0.08,0.00
3,79.17,7.25,0.00,0.00
4,79.11,5.70,0.17 ± 0.23,0.03 ± 0.02
5,93.10,2.17,0.00,0.00
6,68.16,6.07,0.18 ± 0.17,0.25
7,92.36,0.00,0.00,0.00
8,92.30,0.48,0.00,0.00
9,88.12,1.73,0.00,0.00


Algoritmos → (RPD % ± DP),OrderBased,AisleBased,GVNS,Santos e Baldotto (2025)
Instância,,,,
1,71.53,7.20,1.51 ± 0.46,0.00
2,93.20,0.00,0.00,0.00
3,94.98,0.05,0.04 ± 0.10,0.00
4,80.77,4.81,3.00 ± 0.45,0.00
5,94.12,0.00,0.00,0.00
6,83.66,3.44,0.15,0.00
7,86.98,2.92,0.00,0.00
8,94.68 ± 0.02,4.11,0.00,0.00
9,94.92,0.70,0.00,0.21


In [16]:
def tabela_obj(ds):
    d = agg[agg.dataset == ds]
    m = d.pivot(index="instance", columns="algo", values="media")[ORDEM]
    sd = d.pivot(index="instance", columns="algo", values="dp")[ORDEM]
    for x in (m, sd):
        x.index = [int(i.split("_")[1].split(".")[0]) for i in x.index]
    s = m.map("{:.2f}".format) + " ± " + sd.map("{:.2f}".format)
    s = s.mask(sd.abs() < 1e-9, m.map("{:.2f}".format))  # DP zero: mostra so a media
    s.index.name = "Instância"
    s.columns.name = "Algoritmos → (média ± DP)"
    return s

for ds in DATASETS:
    s = tabela_obj(ds)
    titulo = f"Dataset {ds.upper()}: média da função objetivo ± desvio padrão (30 seeds)"
    display(s.style.set_table_styles([
                {"selector": "th", "props": [("text-align", "center"), ("border", "1px solid #999")]},
                {"selector": "td", "props": [("text-align", "center")]}])
             .set_caption(titulo))
    salvar_tabela(s, titulo, f"tabela_obj_{ds}")
    if "latex_tabela" in globals():  # definido na seção de LaTeX (executar antes para gerar o .tex)
        salvar_tex(f"tabela_obj_{ds}", latex_tabela(
            s, ds,
            f"Média da função objetivo $\\pm$ desvio padrão por instância e algoritmo (dataset~{ds.upper()}, 30 sementes).",
            f"tab:obj_{ds}"))


Algoritmos → (média ± DP),OrderBased,AisleBased,GVNS,Santos e Baldotto (2025)
Instância,,,,
1,5.23,15.00,15.00,15.00
2,2.00,2.00,2.00,2.00
3,4.61,8.50,12.00,12.00
4,2.11,3.50,3.50,3.50
5,35.16,172.38,177.88,177.88
6,31.90,691.00,691.00,691.00
7,28.71,392.25,392.25,392.25
8,32.26,146.63,161.88 ± 0.38,162.65
9,2.94,3.04,4.41 ± 0.06,4.42


% ===== tabela_obj_a.tex =====
\begin{table}[H]
  \centering
  \begin{minipage}{\textwidth}
    \caption{\label{tab:obj_a}
      Média da função objetivo $\pm$ desvio padrão por instância e algoritmo (dataset~A, 30 sementes).}
    \resizebox{\textwidth}{!}{
      \begin{tabular}{l | l | l | l | l}
        \hline
        \textbf{Instância} & \textbf{OrderBased} & \textbf{AisleBased} & \textbf{GVNS} & \textbf{Santos e Baldotto (2025)} \\
        \hline
        instance\_0001.txt & 5.23 & 15.00 & 15.00 & 15.00 \\
        \rowcolor{gray!20} instance\_0002.txt & 2.00 & 2.00 & 2.00 & 2.00 \\
        instance\_0003.txt & 4.61 & 8.50 & 12.00 & 12.00 \\
        \rowcolor{gray!20} instance\_0004.txt & 2.11 & 3.50 & 3.50 & 3.50 \\
        instance\_0005.txt & 35.16 & 172.38 & 177.88 & 177.88 \\
        \rowcolor{gray!20} instance\_0006.txt & 31.90 & 691.00 & 691.00 & 691.00 \\
        instance\_0007.txt & 28.71 & 392.25 & 392.25 & 392.25 \\
        \rowcolor{gray!20} instance\_0008.txt & 32.26 & 

Algoritmos → (média ± DP),OrderBased,AisleBased,GVNS,Santos e Baldotto (2025)
Instância,,,,
1,30.29,631.50,631.50,631.50
2,34.74,173.58,180.55 ± 0.08,180.64
3,31.16,138.75,149.60,149.60
4,32.58,147.05,155.67 ± 0.23,155.89 ± 0.02
5,25.39,360.00,368.00,368.00
6,26.75,78.93,83.87 ± 0.17,83.82
7,8.25,108.00,108.00,108.00
8,17.49,226.00,227.10,227.10
9,9.80,81.07,82.50,82.50


% ===== tabela_obj_b.tex =====
\begin{table}[H]
  \centering
  \begin{minipage}{\textwidth}
    \caption{\label{tab:obj_b}
      Média da função objetivo $\pm$ desvio padrão por instância e algoritmo (dataset~B, 30 sementes).}
    \resizebox{\textwidth}{!}{
      \begin{tabular}{l | l | l | l | l}
        \hline
        \textbf{Instância} & \textbf{OrderBased} & \textbf{AisleBased} & \textbf{GVNS} & \textbf{Santos e Baldotto (2025)} \\
        \hline
        instance\_0001.txt & 30.29 & 631.50 & 631.50 & 631.50 \\
        \rowcolor{gray!20} instance\_0002.txt & 34.74 & 173.58 & 180.55 $\pm$ 0.08 & 180.64 \\
        instance\_0003.txt & 31.16 & 138.75 & 149.60 & 149.60 \\
        \rowcolor{gray!20} instance\_0004.txt & 32.58 & 147.05 & 155.67 $\pm$ 0.23 & 155.89 $\pm$ 0.02 \\
        instance\_0005.txt & 25.39 & 360.00 & 368.00 & 368.00 \\
        \rowcolor{gray!20} instance\_0006.txt & 26.75 & 78.93 & 83.87 $\pm$ 0.17 & 83.82 \\
        instance\_0007.txt & 8.25 & 108.00 & 108.00 & 108

Algoritmos → (média ± DP),OrderBased,AisleBased,GVNS,Santos e Baldotto (2025)
Instância,,,,
1,20.17,65.75,69.78 ± 0.46,70.85
2,14.80,217.67,217.67,217.67
3,33.54,667.33,667.37 ± 0.10,667.67
4,38.99,193.00,196.67 ± 0.45,202.75
5,36.95,628.67,628.67,628.67
6,31.36,185.27,191.57,191.86
7,14.99,111.79,115.15,115.15
8,23.33 ± 0.02,420.20,438.22,438.22
9,41.66,814.12,819.88,818.12


% ===== tabela_obj_x.tex =====
\begin{table}[H]
  \centering
  \begin{minipage}{\textwidth}
    \caption{\label{tab:obj_x}
      Média da função objetivo $\pm$ desvio padrão por instância e algoritmo (dataset~X, 30 sementes).}
    \resizebox{\textwidth}{!}{
      \begin{tabular}{l | l | l | l | l}
        \hline
        \textbf{Instância} & \textbf{OrderBased} & \textbf{AisleBased} & \textbf{GVNS} & \textbf{Santos e Baldotto (2025)} \\
        \hline
        instance\_0001.txt & 20.17 & 65.75 & 69.78 $\pm$ 0.46 & 70.85 \\
        \rowcolor{gray!20} instance\_0002.txt & 14.80 & 217.67 & 217.67 & 217.67 \\
        instance\_0003.txt & 33.54 & 667.33 & 667.37 $\pm$ 0.10 & 667.67 \\
        \rowcolor{gray!20} instance\_0004.txt & 38.99 & 193.00 & 196.67 $\pm$ 0.45 & 202.75 \\
        instance\_0005.txt & 36.95 & 628.67 & 628.67 & 628.67 \\
        \rowcolor{gray!20} instance\_0006.txt & 31.36 & 185.27 & 191.57 & 191.86 \\
        instance\_0007.txt & 14.99 & 111.79 & 115.15 & 115.15 \\
   

## Comparação par a par: RPD_A − RPD_B

Para cada dataset, a diferença `RPD_A − RPD_B` (pontos percentuais) por instância. Como o RPD mede a distância ao melhor objetivo, **valor positivo (verde) = B melhor que A**; negativo (vermelho) = A melhor.

In [17]:
from itertools import combinations

PARES = list(combinations(ORDEM, 2))

def tabela_diff(ds):
    rpd = tabela(ds).loc["RPD (%)"]
    diff = pd.DataFrame({f"{a} − {b}": rpd[a] - rpd[b] for a, b in PARES})
    diff.index.name = "Instância"
    diff.columns.name = "RPD_A − RPD_B"
    return diff

def cor(v):
    if pd.isna(v) or abs(v) < 1e-9:
        return ""
    return "color: #1a9c00" if v > 0 else "color: #d00000"

for ds in DATASETS:
    d = tabela_diff(ds)
    display(d.style.format("{:+.2f}").map(cor)
             .set_table_styles([{"selector": "th", "props": [("text-align", "center"), ("border", "1px solid #999")]},
                                {"selector": "td", "props": [("text-align", "center")]}])
             .set_caption(f"Dataset {ds.upper()}: RPD_A − RPD_B (p.p.)"))
    salvar_diff(d, f"Dataset {ds.upper()}: RPD_A − RPD_B (p.p.)", f"diff_todos_pares_{ds}")


RPD_A − RPD_B,OrderBased − AisleBased,OrderBased − GVNS,OrderBased − Santos e Baldotto (2025),AisleBased − GVNS,AisleBased − Santos e Baldotto (2025),GVNS − Santos e Baldotto (2025)
Instância,,,,,,
1,+65.13,+65.13,+65.13,+0.00,+0.00,+0.00
2,+0.00,+0.00,+0.00,+0.00,+0.00,+0.00
3,+32.43,+61.59,+61.59,+29.17,+29.17,+0.00
4,+39.85,+39.85,+39.85,+0.00,+0.00,+0.00
5,+77.14,+80.23,+80.23,+3.09,+3.09,+0.00
6,+95.38,+95.38,+95.38,+0.00,+0.00,+0.00
7,+92.68,+92.68,+92.68,+0.00,+0.00,+0.00
8,+70.19,+79.55,+80.02,+9.36,+9.83,+0.47
9,+2.21,+33.12,+33.38,+30.91,+31.17,+0.26


RPD_A − RPD_B,OrderBased − AisleBased,OrderBased − GVNS,OrderBased − Santos e Baldotto (2025),AisleBased − GVNS,AisleBased − Santos e Baldotto (2025),GVNS − Santos e Baldotto (2025)
Instância,,,,,,
1,+95.20,+95.20,+95.20,+0.00,+0.00,+0.00
2,+76.86,+80.72,+80.77,+3.86,+3.90,+0.05
3,+71.92,+79.17,+79.17,+7.25,+7.25,+0.00
4,+73.41,+78.93,+79.07,+5.53,+5.67,+0.14
5,+90.93,+93.10,+93.10,+2.17,+2.17,+0.00
6,+62.10,+67.98,+67.91,+5.88,+5.81,-0.07
7,+92.36,+92.36,+92.36,+0.00,+0.00,+0.00
8,+91.82,+92.30,+92.30,+0.48,+0.48,+0.00
9,+86.39,+88.12,+88.12,+1.73,+1.73,+0.00


RPD_A − RPD_B,OrderBased − AisleBased,OrderBased − GVNS,OrderBased − Santos e Baldotto (2025),AisleBased − GVNS,AisleBased − Santos e Baldotto (2025),GVNS − Santos e Baldotto (2025)
Instância,,,,,,
1,+64.34,+70.03,+71.53,+5.69,+7.20,+1.51
2,+93.20,+93.20,+93.20,+0.00,+0.00,+0.00
3,+94.93,+94.93,+94.98,+0.00,+0.05,+0.04
4,+75.96,+77.77,+80.77,+1.81,+4.81,+3.00
5,+94.12,+94.12,+94.12,+0.00,+0.00,+0.00
6,+80.22,+83.51,+83.66,+3.29,+3.44,+0.15
7,+84.05,+86.98,+86.98,+2.92,+2.92,+0.00
8,+90.56,+94.68,+94.68,+4.11,+4.11,+0.00
9,+94.22,+94.92,+94.71,+0.70,+0.49,-0.21


## Comparação em sequência: RPD_A − RPD_B

Mesma tabela da comparação anterior, apenas com os pares consecutivos: OrderBased × AisleBased, AisleBased × ILS-VND e Santos e Baldotto (2025) × ILS-VND. Verde (positivo) = B melhor que A.

In [18]:
PARES_SEQ = [("OrderBased", "AisleBased"), ("AisleBased", "ILS-VND"), ("Santos e Baldotto (2025)", "ILS-VND")]

def tabela_diff_seq(ds):
    rpd = tabela(ds).loc["RPD (%)"]
    diff = pd.DataFrame({f"{a} − {b}": rpd[a] - rpd[b] for a, b in PARES_SEQ})
    diff.index.name = "Instância"
    diff.columns.name = "RPD_A − RPD_B"
    return diff

for ds in DATASETS:
    d = tabela_diff_seq(ds)
    display(d.style.format("{:+.2f}").map(cor)
             .set_table_styles([{"selector": "th", "props": [("text-align", "center"), ("border", "1px solid #999")]},
                                {"selector": "td", "props": [("text-align", "center")]}])
             .set_caption(f"Dataset {ds.upper()}: RPD_A − RPD_B, pares em sequência (p.p.)"))
    salvar_diff(d, f"Dataset {ds.upper()}: RPD_A − RPD_B, pares em sequência (p.p.)", f"diff_sequencia_{ds}")


RPD_A − RPD_B,OrderBased − AisleBased,AisleBased − GVNS,Santos e Baldotto (2025) − GVNS
Instância,,,
1,+65.13,+0.00,+0.00
2,+0.00,+0.00,+0.00
3,+32.43,+29.17,+0.00
4,+39.85,+0.00,+0.00
5,+77.14,+3.09,+0.00
6,+95.38,+0.00,+0.00
7,+92.68,+0.00,+0.00
8,+70.19,+9.36,-0.47
9,+2.21,+30.91,-0.26


RPD_A − RPD_B,OrderBased − AisleBased,AisleBased − GVNS,Santos e Baldotto (2025) − GVNS
Instância,,,
1,+95.20,+0.00,+0.00
2,+76.86,+3.86,-0.05
3,+71.92,+7.25,+0.00
4,+73.41,+5.53,-0.14
5,+90.93,+2.17,+0.00
6,+62.10,+5.88,+0.07
7,+92.36,+0.00,+0.00
8,+91.82,+0.48,+0.00
9,+86.39,+1.73,+0.00


RPD_A − RPD_B,OrderBased − AisleBased,AisleBased − GVNS,Santos e Baldotto (2025) − GVNS
Instância,,,
1,+64.34,+5.69,-1.51
2,+93.20,+0.00,+0.00
3,+94.93,+0.00,-0.04
4,+75.96,+1.81,-3.00
5,+94.12,+0.00,+0.00
6,+80.22,+3.29,-0.15
7,+84.05,+2.92,+0.00
8,+90.56,+4.11,+0.00
9,+94.22,+0.70,+0.21


## Tabelas em LaTeX

Gera o código LaTeX de todas as tabelas acima (arquivos `.tex` em `notebooks/resultados/tex/`). Requer `\usepackage{float, graphicx, xcolor, booktabs}` no preâmbulo.


In [19]:
TEX = ROOT / "notebooks" / "resultados" / "tex"
TEX.mkdir(parents=True, exist_ok=True)

def tex_esc(s):
    return str(s).replace("_", r"\_").replace("−", "-")

def latex_tabela(txt, ds, caption, label, cores=None, cabecalho=None):
    """txt: DataFrame de strings (índice = nº da instância). cores: DataFrame opcional de cores xcolor."""
    cols = list(txt.columns)
    n = len(cols)
    L = []
    L.append(r"\begin{table}[H]")
    L.append(r"  \centering")
    L.append(r"  \begin{minipage}{\textwidth}")
    L.append(rf"    \caption{{\label{{{label}}}")
    L.append(rf"      {caption}}}")
    L.append(r"    \resizebox{\textwidth}{!}{")
    L.append(rf"      \begin{{tabular}}{{{' | '.join(['l'] * (n + 1))}}}")
    L.append(r"        \hline")
    L.append(r"        \textbf{Instância} & " + " & ".join(rf"\textbf{{{tex_esc(c)}}}" for c in cols) + r" \\")
    L.append(r"        \hline")
    for i, (idx, row) in enumerate(txt.iterrows()):
        cel = []
        for j, v in enumerate(row):
            v = str(v).replace("±", r"$\pm$")
            cor = cores.iloc[i, j] if cores is not None else ""
            cel.append(rf"\textcolor{{{cor}}}{{{v}}}" if cor else v)
        pre = r"\rowcolor{gray!20} " if i % 2 else ""
        L.append(rf"        {pre}instance\_{int(idx):04d}.txt & " + " & ".join(cel) + r" \\")
    L.append(r"        \hline")
    L.append(r"      \end{tabular}")
    L.append(r"    }")
    L.append(r"    \caption*{\footnotesize Fonte: Elaborado pelo autor, 2026.}")
    L.append(r"  \end{minipage}")
    L.append(r"\end{table}")
    return "\n".join(L)

def cores_diff(d):
    return d.map(lambda v: "green!50!black" if v > 1e-9 else ("red!80!black" if v < -1e-9 else ""))

def salvar_tex(nome, tex):
    (TEX / f"{nome}.tex").write_text(tex + "\n", encoding="utf-8")
    print(f"% ===== {nome}.tex =====\n{tex}\n")

for ds in DATASETS:
    # 1) RPD ± DP
    t = tabela(ds)
    rpd, dp = t.loc["RPD (%)"], t.loc["Desvio Padrão"]
    s = rpd.map("{:.2f}".format) + " ± " + dp.map("{:.2f}".format)
    s = s.mask(dp.abs() < 1e-9, rpd.map("{:.2f}".format))
    salvar_tex(f"tabela_rpd_dp_{ds}", latex_tabela(
        s, ds,
        f"RPD (\\%) $\\pm$ desvio padrão do objetivo por instância e algoritmo (dataset~{ds.upper()}, 30 sementes).",
        f"tab:rpd_dp_{ds}"))

    # 2) Diferenças RPD_A - RPD_B (todos os pares)
    d = tabela_diff(ds)
    salvar_tex(f"diff_todos_pares_{ds}", latex_tabela(
        d.map("{:+.2f}".format), ds,
        f"Diferença $\\mathrm{{RPD}}_A - \\mathrm{{RPD}}_B$ (p.p.) entre todos os pares de algoritmos (dataset~{ds.upper()}). Valores positivos (verde) indicam que B é melhor.",
        f"tab:diff_pares_{ds}", cores_diff(d)))

    # 3) Diferenças em sequência
    d = tabela_diff_seq(ds)
    salvar_tex(f"diff_sequencia_{ds}", latex_tabela(
        d.map("{:+.2f}".format), ds,
        f"Diferença $\\mathrm{{RPD}}_A - \\mathrm{{RPD}}_B$ (p.p.) entre pares consecutivos de algoritmos (dataset~{ds.upper()}). Valores positivos (verde) indicam que B é melhor.",
        f"tab:diff_seq_{ds}", cores_diff(d)))


% ===== tabela_rpd_dp_a.tex =====
\begin{table}[H]
  \centering
  \begin{minipage}{\textwidth}
    \caption{\label{tab:rpd_dp_a}
      RPD (\%) $\pm$ desvio padrão do objetivo por instância e algoritmo (dataset~A, 30 sementes).}
    \resizebox{\textwidth}{!}{
      \begin{tabular}{l | l | l | l | l}
        \hline
        \textbf{Instância} & \textbf{OrderBased} & \textbf{AisleBased} & \textbf{GVNS} & \textbf{Santos e Baldotto (2025)} \\
        \hline
        instance\_0001.txt & 65.13 & 0.00 & 0.00 & 0.00 \\
        \rowcolor{gray!20} instance\_0002.txt & 0.00 & 0.00 & 0.00 & 0.00 \\
        instance\_0003.txt & 61.59 & 29.17 & 0.00 & 0.00 \\
        \rowcolor{gray!20} instance\_0004.txt & 39.85 & 0.00 & 0.00 & 0.00 \\
        instance\_0005.txt & 80.23 & 3.09 & 0.00 & 0.00 \\
        \rowcolor{gray!20} instance\_0006.txt & 95.38 & 0.00 & 0.00 & 0.00 \\
        instance\_0007.txt & 92.68 & 0.00 & 0.00 & 0.00 \\
        \rowcolor{gray!20} instance\_0008.txt & 80.20 & 10.01 & 0.65 $\pm